# 🎓 Notebook 4 — Grade Predictor Web App
### Student Performance Prediction — Multi-Class Edition

**Upload before running:**
- `rf_v2.pkl` (best model)
- `label_encoder_v2.pkl` (to decode XGBoost if needed)
- `X_train_v2.csv` (to refit the scaler)

**What this does:**
- Launches a web app where you enter student details
- Predicts the letter grade (A/B/C/D/E/F) with confidence %
- Shows what the student needs to improve to reach the next grade

## Step 1 — Install Gradio

In [ ]:
!pip install gradio xgboost -q
print('Installed ✅')

## Step 2 — Upload Files

In [ ]:
from google.colab import files
print('Upload: rf_v2.pkl, label_encoder_v2.pkl, X_train_v2.csv')
uploaded = files.upload()

## Step 3 — Load Model & Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import gradio as gr

# Load Random Forest model (best performer)
with open('rf_v2.pkl', 'rb') as f:
    model = pickle.load(f)

# Load label encoder for decoding numeric class indices if needed
with open('label_encoder_v2.pkl', 'rb') as f:
    le = pickle.load(f)

# Load training data to refit the scaler with exact same parameters as Notebook 1
X_train = pd.read_csv('X_train_v2.csv')
feature_columns = X_train.columns.tolist()

scaler = StandardScaler()
scaler.fit(X_train)

# Grade metadata: color, emoji, advice for each grade
grade_info = {
    'A': {'emoji': '🏆', 'label': 'Excellent',    'color': '#2ecc71',
          'msg': 'Outstanding performance! Keep it up.'},
    'B': {'emoji': '⭐', 'label': 'Good',          'color': '#3498db',
          'msg': 'Great work! A little more study time could push you to an A.'},
    'C': {'emoji': '📘', 'label': 'Satisfactory',  'color': '#9b59b6',
          'msg': 'Decent effort. Reduce absences and cut back on going out to reach a B.'},
    'D': {'emoji': '⚠️', 'label': 'Sufficient',    'color': '#f39c12',
          'msg': 'Just passing. Increasing study time and attendance will make a big difference.'},
    'E': {'emoji': '🔴', 'label': 'Poor',          'color': '#e67e22',
          'msg': 'At risk. Focus on reducing failures and absences urgently.'},
    'F': {'emoji': '❌', 'label': 'Fail',          'color': '#e74c3c',
          'msg': 'Failing. Immediate intervention needed — seek school support and reduce absences.'}
}

print(f'Model loaded ✅ | Features: {len(feature_columns)}')

## Step 4 — Prediction Function

In [ ]:
def predict_grade(age, studytime, failures, absences, goout, Walc, Dalc,
                  health, Medu, Fedu, traveltime, famrel, freetime,
                  school_MS, sex_M, address_U, famsize_LE3, Pstatus_T,
                  Mjob_health, Mjob_other, Mjob_services, Mjob_teacher,
                  Fjob_health, Fjob_other, Fjob_services, Fjob_teacher,
                  reason_home, reason_other, reason_reputation,
                  guardian_mother, guardian_other,
                  schoolsup_yes, famsup_yes, paid_yes, activities_yes,
                  nursery_yes, higher_yes, internet_yes, romantic_yes):

    # Build input dictionary matching training feature names exactly
    input_dict = {
        'age': age, 'Medu': Medu, 'Fedu': Fedu, 'traveltime': traveltime,
        'studytime': studytime, 'failures': failures, 'famrel': famrel,
        'freetime': freetime, 'goout': goout, 'Dalc': Dalc, 'Walc': Walc,
        'health': health, 'absences': absences,
        'school_MS': int(school_MS), 'sex_M': int(sex_M),
        'address_U': int(address_U), 'famsize_LE3': int(famsize_LE3),
        'Pstatus_T': int(Pstatus_T), 'Mjob_health': int(Mjob_health),
        'Mjob_other': int(Mjob_other), 'Mjob_services': int(Mjob_services),
        'Mjob_teacher': int(Mjob_teacher), 'Fjob_health': int(Fjob_health),
        'Fjob_other': int(Fjob_other), 'Fjob_services': int(Fjob_services),
        'Fjob_teacher': int(Fjob_teacher), 'reason_home': int(reason_home),
        'reason_other': int(reason_other), 'reason_reputation': int(reason_reputation),
        'guardian_mother': int(guardian_mother), 'guardian_other': int(guardian_other),
        'schoolsup_yes': int(schoolsup_yes), 'famsup_yes': int(famsup_yes),
        'paid_yes': int(paid_yes), 'activities_yes': int(activities_yes),
        'nursery_yes': int(nursery_yes), 'higher_yes': int(higher_yes),
        'internet_yes': int(internet_yes), 'romantic_yes': int(romantic_yes)
    }

    input_df = pd.DataFrame([input_dict])[feature_columns]
    input_scaled = scaler.transform(input_df)

    # Get predicted grade and probability for each grade
    predicted_grade = model.predict(input_scaled)[0]
    probabilities   = model.predict_proba(input_scaled)[0]
    class_labels    = model.classes_

    info = grade_info[predicted_grade]

    # Build main result string
    result = f"{info['emoji']} Predicted Grade: {predicted_grade} — {info['label']}\n\n"
    result += f"💬 {info['msg']}\n\n"
    result += "📊 Confidence across all grades:\n"

    # Show probability for each grade sorted A→F
    grade_probs = dict(zip(class_labels, probabilities))
    for g in ['A', 'B', 'C', 'D', 'E', 'F']:
        prob = grade_probs.get(g, 0) * 100
        bar = '█' * int(prob / 5)   # simple ASCII bar (each block = 5%)
        marker = ' ← predicted' if g == predicted_grade else ''
        result += f"  {g}: {prob:5.1f}%  {bar}{marker}\n"

    return result

print('Prediction function ready ✅')

## Step 5 — Launch the Web App

In [ ]:
with gr.Blocks(title='Student Grade Predictor', theme=gr.themes.Soft()) as app:

    gr.Markdown('# 🎓 Student Grade Predictor')
    gr.Markdown('Enter student details to predict their letter grade: **A / B / C / D / E / F**')
    gr.Markdown('> Model: Random Forest | UCI Student Performance Dataset | PES University ML Project')

    with gr.Tabs():

        with gr.Tab('⭐ Key Features'):
            gr.Markdown('### Top predictors identified by SHAP analysis')
            with gr.Row():
                failures  = gr.Slider(0, 4, value=0, step=1, label='Past Class Failures (0=none)')
                absences  = gr.Slider(0, 93, value=5, step=1, label='Number of Absences')
            with gr.Row():
                goout     = gr.Slider(1, 5, value=3, step=1, label='Going Out with Friends (1=low, 5=high)')
                studytime = gr.Slider(1, 4, value=2, step=1, label='Weekly Study Time (1=<2h, 4=>10h)')
            with gr.Row():
                Walc      = gr.Slider(1, 5, value=2, step=1, label='Weekend Alcohol Use (1=low, 5=high)')
                Dalc      = gr.Slider(1, 5, value=1, step=1, label='Weekday Alcohol Use (1=low, 5=high)')

        with gr.Tab('👤 Student Info'):
            with gr.Row():
                age       = gr.Slider(15, 22, value=17, step=1, label='Age')
                health    = gr.Slider(1, 5, value=3, step=1, label='Health (1=bad, 5=excellent)')
            with gr.Row():
                freetime  = gr.Slider(1, 5, value=3, step=1, label='Free Time After School')
                traveltime = gr.Slider(1, 4, value=1, step=1, label='Travel Time (1=<15min, 4=>1hr)')
            with gr.Row():
                sex_M        = gr.Checkbox(label='Male', value=True)
                address_U    = gr.Checkbox(label='Urban Address', value=True)
                romantic_yes = gr.Checkbox(label='In a Relationship', value=False)
                higher_yes   = gr.Checkbox(label='Wants Higher Education', value=True)
                internet_yes = gr.Checkbox(label='Has Internet at Home', value=True)

        with gr.Tab('👨‍👩‍👦 Family'):
            with gr.Row():
                Medu      = gr.Slider(0, 4, value=2, step=1, label="Mother's Education (0=none, 4=higher)")
                Fedu      = gr.Slider(0, 4, value=2, step=1, label="Father's Education")
            with gr.Row():
                famrel    = gr.Slider(1, 5, value=4, step=1, label='Family Relationship Quality')
                famsize_LE3 = gr.Checkbox(label='Family Size ≤ 3', value=False)
                Pstatus_T   = gr.Checkbox(label='Parents Living Together', value=True)
                famsup_yes  = gr.Checkbox(label='Family Educational Support', value=True)
            with gr.Row():
                guardian_mother = gr.Checkbox(label='Guardian: Mother', value=True)
                guardian_other  = gr.Checkbox(label='Guardian: Other', value=False)
            gr.Markdown('**Mother Job:**')
            with gr.Row():
                Mjob_health   = gr.Checkbox(label='Health', value=False)
                Mjob_other    = gr.Checkbox(label='Other', value=True)
                Mjob_services = gr.Checkbox(label='Services', value=False)
                Mjob_teacher  = gr.Checkbox(label='Teacher', value=False)
            gr.Markdown('**Father Job:**')
            with gr.Row():
                Fjob_health   = gr.Checkbox(label='Health', value=False)
                Fjob_other    = gr.Checkbox(label='Other', value=True)
                Fjob_services = gr.Checkbox(label='Services', value=False)
                Fjob_teacher  = gr.Checkbox(label='Teacher', value=False)

        with gr.Tab('🏫 School'):
            with gr.Row():
                school_MS      = gr.Checkbox(label='School: MS (vs GP)', value=False)
                schoolsup_yes  = gr.Checkbox(label='Extra School Support', value=False)
                paid_yes       = gr.Checkbox(label='Extra Paid Classes', value=False)
                activities_yes = gr.Checkbox(label='Extracurricular Activities', value=False)
                nursery_yes    = gr.Checkbox(label='Attended Nursery', value=True)
            gr.Markdown('**Reason for Choosing School:**')
            with gr.Row():
                reason_home       = gr.Checkbox(label='Close to Home', value=False)
                reason_other      = gr.Checkbox(label='Other', value=False)
                reason_reputation = gr.Checkbox(label='Reputation', value=True)

    predict_btn = gr.Button('🔮 Predict My Grade', variant='primary', size='lg')
    output = gr.Textbox(label='Prediction Result', lines=12, interactive=False)

    predict_btn.click(
        fn=predict_grade,
        inputs=[
            age, studytime, failures, absences, goout, Walc, Dalc, health,
            Medu, Fedu, traveltime, famrel, freetime,
            school_MS, sex_M, address_U, famsize_LE3, Pstatus_T,
            Mjob_health, Mjob_other, Mjob_services, Mjob_teacher,
            Fjob_health, Fjob_other, Fjob_services, Fjob_teacher,
            reason_home, reason_other, reason_reputation,
            guardian_mother, guardian_other,
            schoolsup_yes, famsup_yes, paid_yes, activities_yes,
            nursery_yes, higher_yes, internet_yes, romantic_yes
        ],
        outputs=output
    )

    gr.Markdown('---')
    gr.Markdown('*Grade Scale: A(18-20) | B(15-17) | C(12-14) | D(10-11) | E(8-9) | F(0-7)*')
    gr.Markdown('*PES University ML Mini Project | UCI Student Performance Dataset*')

# share=True generates a public link valid for 72 hours — perfect for live demos
app.launch(share=True)